In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score, RocCurveDisplay
)

from sklearn.linear_model import LogisticRegression, LassoCV, Lasso
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.inspection import permutation_importance

In [ ]:
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

df = pd.read_csv("/kaggle/input/datasets/thedevastator/unlocking-mysteries-of-bigfoot-through-sightings/bfro_reports_geocoded.csv")

print("Shape of raw dataset:", df.shape)
df.head()

In [ ]:
print("Columns:")
print(df.columns.tolist())

print("\nClassification counts:")
print(df["classification"].value_counts(dropna=False))

In [ ]:
df = df[df["classification"].isin(["Class A", "Class B"])].copy()

df["target"] = df["classification"].map({
    "Class A": 1,
    "Class B": 0
})

print("Shape after filtering:", df.shape)
print(df["classification"].value_counts())

In [ ]:
drop_cols = [
    "number",
    "title",
    "location_details",
    "summary",
    "geohash",
    "classification",
    "observed"
]

existing_drop_cols = [col for col in drop_cols if col in df.columns]
df = df.drop(columns=existing_drop_cols)

#All of the Xs and Ys
X_full = df.drop(columns=["target"])
y_full = df["target"]

#numeric columns and categorical columns
numeric_cols = X_full.select_dtypes(include=np.number).columns.tolist()
categorical_cols = X_full.select_dtypes(exclude=np.number).columns.tolist()

print("Remaining columns:")
print(df.columns.tolist())
print("Shape after filtering:", df.shape)

In [ ]:
missing_pct = df.isnull().mean().sort_values(ascending=False) * 100
missing_pct = missing_pct[missing_pct > 0]

plt.figure(figsize=(10, 6))
missing_pct.plot(kind="bar")
plt.title("Missing Value Percentage by Column")
plt.ylabel("Percent Missing")
plt.show()

In [ ]:
#Missing value cleaning: drop NaN rows
df_drop = df.copy()

# Drop any rows with missing values
df_drop = df_drop.dropna()

X_drop = df_drop.drop(columns=["target"])
y_drop = df_drop["target"]

print("Shape after dropping NaN:", df_drop.shape)

In [ ]:
#Missing value cleaning: Replace Nan with Median
df_median = df.copy()

# Replace with Median in numeric column
for col in numeric_cols:
    median_val = df_median[col].median()
    df_median[col] = df_median[col].fillna(median_val)

# Replace with the first category for categorical column
for col in categorical_cols:
    mode_val = df_median[col].mode(dropna=True)
    if len(mode_val) > 0:
        df_median[col] = df_median[col].fillna(mode_val.iloc[0])

print("Remaining NaNs after imputation:")
print(df_median.isnull().sum().sum())

In [ ]:
# plt.figure()
# sns.countplot(data=df, x="target")
# plt.xticks([0, 1], ["Class B", "Class A"])
# plt.title("Class Distribution")
# plt.show()

#sighting class count(drop NaN)
plt.figure()
sns.countplot(data=df_drop, x="target")
plt.xticks([0, 1], ["Class B", "Class A"])
plt.title("Class Distribution(drop NaN)")
plt.show()

#sighting class count(replace NaN with median)
plt.figure()
sns.countplot(data=df_median, x="target")
plt.xticks([0, 1], ["Class B", "Class A"])
plt.title("Class Distribution(Median replace)")
plt.show()

In [ ]:
#Sightings by Season and Class(drop NaN)
if "season" in df_drop.columns:
    plt.figure()
    sns.countplot(data=df_drop, x="season", hue="target")
    plt.legend(title="Class", labels=["Class B", "Class A"])
    plt.title("Sightings by Season and Class(drop NaN)")
    plt.show()

#Sightings by Season and Classreplace NaN with median)
if "season" in df_median.columns:
    plt.figure()
    sns.countplot(data=df_median, x="season", hue="target")
    plt.legend(title="Class", labels=["Class B", "Class A"])
    plt.title("Sightings by Season and Class(replace NaN by median)")
    plt.show()

In [ ]:
#Top 10 States(drop NaN)
if "state" in df_drop.columns:
    top_states = df_drop["state"].value_counts().head(10).index
    plt.figure(figsize=(12, 6))
    sns.countplot(
        data=df_drop[df_drop["state"].isin(top_states)],
        x="state",
        hue="target",
        order=top_states
    )
    plt.legend(title="Class", labels=["Class B", "Class A"])
    plt.title("Top 10 States by Number of Sightings(drop NaN)")
    plt.show()

#Top 10 States(replace NaN with median)
if "state" in df_median.columns:
    top_states = df_median["state"].value_counts().head(10).index
    plt.figure(figsize=(12, 6))
    sns.countplot(
        data=df_median[df_median["state"].isin(top_states)],
        x="state",
        hue="target",
        order=top_states
    )
    plt.legend(title="Class", labels=["Class B", "Class A"])
    plt.title("Top 10 States by Number of Sightings(replace NaN by median)")
    plt.show()

In [ ]:
#Correlation heatmap
if len(numeric_cols) > 1:
    plt.figure(figsize=(12, 8))
    corr = df_drop[numeric_cols + ["target"]].corr(numeric_only=True)
    sns.heatmap(corr, cmap="coolwarm", center=0)
    plt.title("Correlation Heatmap(drop NaN)")
    plt.show()

#Correlation heatmap(replace with median)
if len(numeric_cols) > 1:
    plt.figure(figsize=(12, 8))
    corr = df_median[numeric_cols + ["target"]].corr(numeric_only=True)
    sns.heatmap(corr, cmap="coolwarm", center=0)
    plt.title("Correlation Heatmap(replace NaN with median)")
    plt.show()

In [ ]:
#X = df_median.drop(columns=["target", "county", "state", "season", "latitude", "longitude","date"] )
X = df_median.drop(columns=["target"] )
y = df_median["target"]

# One-hot encoding
X_encoded = pd.get_dummies(X, drop_first=True)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Logistic Regression
log_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=5000))
])

log_model.fit(X_train, y_train)

y_pred = log_model.predict(X_test)

model = log_model.named_steps["model"]

coef_df = pd.DataFrame({
    "feature": X_encoded.columns,
    "coefficient": model.coef_[0]
})

coef_df["abs_coef"] = coef_df["coefficient"].abs()
coef_df = coef_df.sort_values("abs_coef", ascending=False)

coef_df.head(20)


In [ ]:
#Plot top factors influencing CLass A and Class B
top_features = coef_df.head(15)

plt.figure(figsize=(10, 6))
sns.barplot(data=top_features, y="feature", x="coefficient")
plt.title("Top Factors Influencing Class A vs Class B")
plt.axvline(0, color="black")
plt.show()

In [ ]:
#Decision tree
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

tree = DecisionTreeClassifier(
    max_depth=3,          
    min_samples_leaf=50,  # avoid overfitting
    random_state=42
)

tree.fit(X_train, y_train)

y_pred = tree.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))

In [ ]:
from sklearn.tree import plot_tree

plt.figure(figsize=(20, 10))

plot_tree(
    tree,
    feature_names=X_encoded.columns,
    class_names=["Class B", "Class A"],
    filled=True,
    rounded=True,
    fontsize=8
)

plt.title("Decision Tree for Class A vs Class B")
plt.show()

In [ ]:
# Keep only rows with valid latitude and longitude
geo_df = df.copy()

geo_df = geo_df.dropna(subset=["latitude", "longitude", "target"]).copy()

print("Shape of geo_df:", geo_df.shape)
geo_df[["latitude", "longitude", "target"]].head()

In [ ]:
plt.figure(figsize=(10, 6))

sns.scatterplot(
    data=geo_df,
    x="longitude",
    y="latitude",
    hue="target",
    palette={0: "blue", 1: "red"},
    alpha=0.7
)

plt.title("Bigfoot Sightings by Latitude and Longitude")
plt.xlabel("Longitude")
plt.ylabel("Latitude")

# Rename legend labels
handles, labels = plt.gca().get_legend_handles_labels()
plt.legend(handles, ["Class B", "Class A"], title="Classification")

plt.show()

In [ ]:
X_geo = geo_df[["longitude", "latitude"]]
y_geo = geo_df["target"]

X_geo_train, X_geo_test, y_geo_train, y_geo_test = train_test_split(
    X_geo, y_geo,
    test_size=0.2,
    random_state=42,
    stratify=y_geo
)

print("Train shape:", X_geo_train.shape)
print("Test shape:", X_geo_test.shape)

In [ ]:
geo_svm_pipeline = Pipeline(steps=[
    ("scaler", StandardScaler()),
    ("svm", SVC(kernel="rbf", C=1.0, gamma="scale"))
])

geo_svm_pipeline.fit(X_geo_train, y_geo_train)

y_geo_pred = geo_svm_pipeline.predict(X_geo_test)

print("Accuracy :", accuracy_score(y_geo_test, y_geo_pred))
print("Precision:", precision_score(y_geo_test, y_geo_pred))
print("Recall   :", recall_score(y_geo_test, y_geo_pred))
print("F1 Score :", f1_score(y_geo_test, y_geo_pred))

In [ ]:
# Create a mesh grid over the longitude-latitude space
x_min, x_max = X_geo["longitude"].min() - 1, X_geo["longitude"].max() + 1
y_min, y_max = X_geo["latitude"].min() - 1, X_geo["latitude"].max() + 1

xx, yy = np.meshgrid(
    np.linspace(x_min, x_max, 300),
    np.linspace(y_min, y_max, 300)
)

grid_points = pd.DataFrame({
    "longitude": xx.ravel(),
    "latitude": yy.ravel()
})

Z = geo_svm_pipeline.predict(grid_points)
Z = Z.reshape(xx.shape)

plt.figure(figsize=(12, 8))

# Decision regions
plt.contourf(xx, yy, Z, alpha=0.25, levels=[-0.5, 0.5, 1.5], cmap="coolwarm")

# Scatter plot of actual points
sns.scatterplot(
    data=geo_df,
    x="longitude",
    y="latitude",
    hue="target",
    palette={0: "blue", 1: "red"},
    edgecolor="black",
    alpha=0.7
)

plt.title("SVM Decision Boundary Using Latitude and Longitude")
plt.xlabel("Longitude")
plt.ylabel("Latitude")

# Custom legend labels
handles, labels = plt.gca().get_legend_handles_labels()
plt.legend(handles, ["Class B", "Class A"], title="Classification")

plt.show()

In [ ]:
# Decision function values for contour lines
Z_decision = geo_svm_pipeline.decision_function(grid_points)
Z_decision = Z_decision.reshape(xx.shape)

plt.figure(figsize=(12, 8))

# Background decision regions
plt.contourf(xx, yy, Z, alpha=0.2, levels=[-0.5, 0.5, 1.5], cmap="coolwarm")

# Decision boundary and margins
plt.contour(
    xx, yy, Z_decision,
    levels=[-1, 0, 1],
    linestyles=["--", "-", "--"],
    colors="black"
)

sns.scatterplot(
    data=geo_df,
    x="longitude",
    y="latitude",
    hue="target",
    palette={0: "blue", 1: "red"},
    edgecolor="black",
    alpha=0.7
)

plt.title("SVM Decision Boundary and Margins (Latitude vs Longitude)")
plt.xlabel("Longitude")
plt.ylabel("Latitude")

handles, labels = plt.gca().get_legend_handles_labels()
plt.legend(handles, ["Class B", "Class A"], title="Classification")

plt.show()